# Missing values, with TabPFN

A hole in the table is the most common defect in real data. This notebook walks the classical
toolkit for handling it, shows how much of that toolkit TabPFN makes unnecessary, and isolates the
two places where it still matters. Every experiment is a small, self-contained synthetic
construction, so the behaviour we observe is real but the setup stays easy to read.

In [10]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from tabpfn import TabPFNClassifier
from tqdm import tqdm
from tabulate import tabulate

def printdf(df, nrows=5, showindex=True): print(tabulate(df.head(nrows), headers='keys', tablefmt='psql', showindex=showindex))

def auc(model, X_train, y_train, X_test, y_test):
    """Fit a model and return its test ROC AUC, rounded."""
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    return round(roc_auc_score(y_test, probabilities), 4)

## How we usually handle missing values

The classical toolkit, in rough order of effort:

- **Drop** rows or columns when the gaps are few and uninformative.
- **Fill** with the median or mode, or a group-wise statistic (learned on train only).
- **Flag** with a missing-indicator column, because absence can be predictive (especially MNAR).
- **Sentinel-encode** (`-9999`) so a tree can split missing into its own branch.
- **Model-based imputation** (MICE, KNN) to reconstruct a column from the others.
- **For tree models, do not impute**: XGBoost, LightGBM and CatBoost take NaNs natively and learn
  where to route them.

Which one you reach for depends on the model, and on *why* the value is missing (MCAR, MAR, MNAR).

## Why is it missing? MCAR, MAR, MNAR

Before choosing a fix, the textbook asks *why* a value is missing, because that decides whether the
gap is noise or signal:

- **MCAR (Missing Completely At Random).** No pattern at all, e.g. a sensor that randomly drops
  readings. That a value is missing tells you nothing.
- **MAR (Missing At Random).** The gap depends on *other observed* columns, e.g. older customers
  skip the income field. You can predict the missingness from what you do observe, so it is
  recoverable.
- **MNAR (Missing Not At Random).** The gap depends on the *missing value itself*, e.g. high
  earners decline to report income. The missingness carries information you cannot fully recover
  from the other columns.

The upshot: under MAR and MNAR, "this value is missing" is itself a feature. That is what a
missing-indicator captures, and why imputing-and-forgetting can throw signal away.

In [11]:
# Three ways a column could go missing. We measure how much "this value is missing"
# correlates with the label under each.
rng = np.random.RandomState(0)
n = 6000

x = rng.randn(n)
other = rng.randn(n)
label = (rng.rand(n) < 1 / (1 + np.exp(-(x + other)))).astype(int)   # label depends on x and other

is_missing = {
    "MCAR (random)":            rng.rand(n) < 0.3,   # no pattern
    "MAR  (depends on other)":  other > 0.5,         # depends on an observed column
    "MNAR (depends on x)":      x > 0.5,             # depends on the missing value itself
}

correlations = {}
for name, mask in is_missing.items():
    correlations[name] = round(abs(np.corrcoef(mask.astype(int), label)[0, 1]), 3)

printdf(pd.DataFrame(correlations.items(), columns=["missing_type", "corr(is_missing, label)"]))

+----+-------------------------+---------------------------+
|    | missing_type            |   corr(is_missing, label) |
|----+-------------------------+---------------------------|
|  0 | MCAR (random)           |                     0.014 |
|  1 | MAR  (depends on other) |                     0.266 |
|  2 | MNAR (depends on x)     |                     0.267 |
+----+-------------------------+---------------------------+


MCAR's indicator is uninformative (~0); under MAR and MNAR the missing-flag correlates with the
label. The difference between those two: MAR's signal is recoverable from `other` (a column you
still observe), while MNAR's lives in the value you can no longer see. That distinction is exactly
where an indicator earns or wastes its place.

## What TabPFN does

Leave the NaNs in. Inside the model, before the forward pass, every missing cell is (1) filled with
that feature's **mean over the training rows**, and (2) recorded in a **per-feature
missing-indicator channel** that the network reads. So TabPFN performs *mean-impute-plus-indicator*
by default, the move strong neural baselines do by hand. Most of the checklist above is already
inside the model. The rest of this notebook asks: which parts does that make redundant, and where
does the toolkit still earn its keep?

In [13]:
# TabPFN accepts NaNs directly. No imputation step needed.
demo = pd.DataFrame(np.random.RandomState(0).randn(200, 3), columns=["a", "b", "c"])
demo.loc[:60, "a"] = np.nan                  # put some holes in column 'a'
demo_label = (demo["b"] > 0).astype(int)

model = TabPFNClassifier(random_state=0)
model.fit(demo, demo_label)                  # fits fine, NaNs and all
model.predict_proba(demo)[:5, 1]             # probabilities for the first five rows

array([9.9998701e-01, 9.9998701e-01, 1.5486538e-04, 9.9990189e-01,
       9.9980223e-01], dtype=float32)

### Can you change how TabPFN handles missing values?

Short answer: no. The mean-fill and the indicator channel are baked into the v3 architecture and
its pretrained weights, not exposed as settings. No constructor argument and no `inference_config`
field controls feature imputation. (The two parameters that touch missing values do so only
indirectly: `categorical_features_indices`, which decides whether a hole-bearing column is encoded
as categorical or numeric, and `inference_config`, which sets the scaling the filled value then
passes through.)

In [14]:
import inspect

# 1) The constructor: no argument is about missing values or imputation.
constructor_params = [p for p in inspect.signature(TabPFNClassifier.__init__).parameters if p != "self"]
print("constructor parameters:")
print(" ", constructor_params)

# 2) Confirm the loaded model is v3, with the indicator channel on.
fitted = TabPFNClassifier(random_state=0).fit(demo, demo_label)
print("\nloaded model:", type(fitted.model_).__name__,
      "| use_nan_indicators =", fitted.model_.use_nan_indicators)

# 3) The preprocessing knobs (inference_config): none controls feature imputation.
config_fields = [f for f in dir(fitted.inference_config_) if f.isupper()]
imputation_knobs = [f for f in config_fields if any(k in f for k in ("IMPUT", "FILL"))]
print("\ninference_config fields that control feature imputation:", imputation_knobs or "none")

constructor parameters:
  ['n_estimators', 'categorical_features_indices', 'softmax_temperature', 'balance_probabilities', 'average_before_softmax', 'model_path', 'device', 'ignore_pretraining_limits', 'inference_precision', 'fit_mode', 'memory_saving_mode', 'random_state', 'n_jobs', 'n_preprocessing_jobs', 'inference_config', 'differentiable_input', 'eval_metric', 'tuning_config', 'show_progress_bar']

loaded model: TabPFNV3 | use_nan_indicators = True

inference_config fields that control feature imputation: none


## Do the classical fills even matter?

A useful numeric column `a` is 40% missing (completely at random), and it varies by an observed
segment `seg` (think product or region). We try the classical fills, mean, median, an out-of-range
sentinel, a segment-wise mean, and mean-plus-indicator, against just leaving the NaN in. (Mode is
the same idea for a categorical column: fill with the most frequent level.) How much does the
choice move each model?

In [15]:
# Build a small dataset: 'a' depends on a segment/group, the label depends on 'a' and 'b'.
rng = np.random.RandomState(0)
n = 6000

seg = rng.randint(0, 4, n)                          # segment 0-3 (e.g. region), observed
segment_value = np.array([4.0, -2.0, 6.0, -1.0])    # each segment's typical 'a' (jumps around)
a = segment_value[seg] + rng.randn(n)               # 'a' = its segment's value + noise
b = rng.randn(n)                                    # a second predictor

score = 0.6 * a + b
label = (rng.rand(n) < 1 / (1 + np.exp(-score))).astype(int)   # 0/1 label from a sigmoid of score

data = pd.DataFrame({"a": a, "b": b, "seg": seg.astype(float)})
data.loc[rng.rand(n) < 0.40, "a"] = np.nan          # knock out 40% of 'a', at random

train, test = data.iloc[:n // 2], data.iloc[n // 2:]
y_train, y_test = label[:n // 2], label[n // 2:]

In [ ]:
# Fill values, all computed on the TRAIN half only (no leakage).
train_mean = train["a"].mean()
train_median = train["a"].median()
segment_mean = train.groupby("seg")["a"].mean()     # mean of 'a' within each segment

def filled(frame, method):
    """Return a copy of `frame` with column 'a' handled by the chosen method."""
    out = frame.copy()
    if method == "mean":
        out["a"] = out["a"].fillna(train_mean)
    elif method == "median":
        out["a"] = out["a"].fillna(train_median)
    elif method == "sentinel":
        out["a"] = out["a"].fillna(-9999)
    elif method == "group":
        out["a"] = out["a"].fillna(out["seg"].map(segment_mean))
    elif method == "indicator":
        out["a_missing"] = out["a"].isna().astype(int)
        out["a"] = out["a"].fillna(train_mean)
    # "native" is not handled, so the NaN is left in place
    return out

In [ ]:
methods = ["native", "mean", "median", "sentinel", "group", "indicator"]
models = {
    "TabPFN":   lambda: TabPFNClassifier(random_state=0),
    "LightGBM": lambda: LGBMClassifier(verbose=-1),
    "Linear":   lambda: Pipeline([("impute", SimpleImputer()),
                                  ("scale", StandardScaler()),
                                  ("logreg", LogisticRegression(max_iter=2000))]),
}

scores = {}
for model_name, make_model in models.items():
    row = {}
    for method in methods:
        row[method] = auc(make_model(), filled(train, method), y_train, filled(test, method), y_test)
    scores[model_name] = row

pd.DataFrame(scores).T

Read across each row. **TabPFN and LightGBM barely move**: they read the segment directly, so how
you fill `a` hardly matters. The **linear model is the sensitive one**: the segment-wise `group`
fill helps it (it cannot use `seg` non-monotonically on its own), and the `-9999` sentinel hurts it
(a giant value wrecks a linear fit, while a tree just splits it off). The `indicator` adds nothing
here, because this gap is MCAR, there is no missingness signal to capture. That is the grounding in
one table: for a flexible model the fill is mostly a non-decision; the care goes into weak models
and into *informative* missingness, which is where the rest of the notebook goes.

To check that group-fill's benefit is entirely about *access to the grouping*, run the same
comparison but drop `seg` from the columns the models see. The fill still uses `seg` to compute the
group mean; the models just no longer get `seg` as a feature, so the only way the segment can reach
them is through the `group` fill.

In [ ]:
# Same comparison, but 'seg' is removed from the features (group-fill still uses it to fill 'a').
scores_no_seg = {}
for model_name, make_model in models.items():
    row = {}
    for method in methods:
        train_f = filled(train, method).drop(columns=["seg"])
        test_f = filled(test, method).drop(columns=["seg"])
        row[method] = auc(make_model(), train_f, y_train, test_f, y_test)
    scores_no_seg[model_name] = row

pd.DataFrame(scores_no_seg).T

Now `group` jumps for **every** model (LightGBM and TabPFN both gain roughly +0.07), while the
other fills stay flat. With no `seg` column, group-fill is the only channel for the segment, so it
is the only fill that carries information. The takeaway, stated once: **group-fill helps exactly
when the model cannot already see the grouping.** A flexible model that has the segment column gets
nothing from group-filling; strip the column and it becomes the best fill there is.

## The toolkit mostly collapses

A classic reason to impute carefully is that a missing column can be *reconstructed* from the
others (model-based imputation). The next few cells build exactly that, with a few small
helpers:

- `make_data(make_Z)` builds a table of random columns `x0..x9` plus a column `Z` that is a chosen
  function of them, with the label depending on `Z`. So `Z` is useful, and reconstructable.
- `hide_half_of_Z(...)` knocks out half of `Z` at random.
- `model_based_impute(...)` trains a small regressor to predict `Z` from the other columns and fills the
  holes with its predictions.
- `compare_imputations(...)` scores four ways of handling the holes (the true `Z`, leaving the NaN,
  mean-fill, and MICE) across the three models.

In [ ]:
def make_data(make_Z, n=8000, seed=0):
    """Build a table where column Z is a function of x0..x9, and the label depends on Z."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, 10)

    signal = make_Z(X)                              # Z's underlying signal, built from x0..x9
    signal = (signal - signal.mean()) / signal.std()
    Z = signal + 0.5 * rng.randn(n)                 # the actual column: signal plus noise

    label = (rng.rand(n) < 1 / (1 + np.exp(-2.5 * signal))).astype(int)

    columns = [f"x{i}" for i in range(10)] + ["Z"]
    table = pd.DataFrame(np.column_stack([X, Z]), columns=columns)
    return table, label

def hide_half_of_Z(table, seed):
    """Set half of column Z to NaN, at random."""
    table = table.copy()
    holes = np.random.RandomState(seed).rand(len(table)) < 0.5
    table.loc[holes, "Z"] = np.nan
    return table

def model_based_impute(train, test):
    """Model-based imputation: predict Z from the other columns, fill the holes with the prediction."""
    other_columns = [c for c in train.columns if c != "Z"]
    observed = train["Z"].notna()

    regressor = LGBMRegressor(n_estimators=400, verbose=-1)
    regressor.fit(train.loc[observed, other_columns], train.loc[observed, "Z"])

    train, test = train.copy(), test.copy()
    for frame in (train, test):
        missing = frame["Z"].isna()
        frame.loc[missing, "Z"] = regressor.predict(frame.loc[missing, other_columns])
    return train, test

In [ ]:
def compare_imputations(table, label):
    """Hide half of Z, then score four ways of handling it across the three models."""
    half = len(table) // 2
    train, test = table.iloc[:half], table.iloc[half:]
    y_train, y_test = label[:half], label[half:]

    train_holes = hide_half_of_Z(train, seed=1)
    test_holes = hide_half_of_Z(test, seed=2)
    column_means = train_holes.mean()

    variants = {
        "oracle (true Z)": (train, test),                                       # never lost Z
        "native (NaN)":    (train_holes, test_holes),                           # leave the holes
        "mean-impute":     (train_holes.fillna(column_means), test_holes.fillna(column_means)),
        "model-based":            model_based_impute(train_holes, test_holes),                # rebuild Z
    }
    models = {
        "TabPFN":   lambda: TabPFNClassifier(random_state=0),
        "LightGBM": lambda: LGBMClassifier(verbose=-1),
        "Linear":   lambda: Pipeline([("impute", SimpleImputer()),
                                      ("scale", StandardScaler()),
                                      ("logreg", LogisticRegression(max_iter=2000))]),
    }

    scores = {}
    for model_name, make_model in models.items():
        row = {}
        for variant_name, (train_variant, test_variant) in variants.items():
            row[variant_name] = auc(make_model(), train_variant, y_train, test_variant, y_test)
        scores[model_name] = row
    return pd.DataFrame(scores).T

In [ ]:
# An easy Z: a small combination of three columns.
def easy_Z(X):
    return X[:, 0] * X[:, 1] + X[:, 2] ** 2

table, label = make_data(easy_Z)
compare_imputations(table, label)

For TabPFN and LightGBM every option ties: leaving the hole, mean-filling, or carefully rebuilding
`Z` all land within noise, and all match the *oracle* that never lost `Z`. They reconstruct what
they need from the other columns internally. Model-based imputation only rescues the **linear** model, which cannot
learn the combination itself. Model-based imputation is a crutch for weak models.

(We also checked hand-built indicators and `-9999` sentinels: for TabPFN both land within noise
too. It is forgiving about how you encode a hole.)

In [ ]:
# A hard Z: a high-order, nonlinear combination of several columns.
def hard_Z(X):
    return (X[:, 0] * X[:, 1] * X[:, 2]
            + 1.5 * np.sin(3 * X[:, 3]) * np.cos(3 * X[:, 4])
            + 2.0 * ((X[:, 5] > 0) & (X[:, 6] > 0))
            - X[:, 7] ** 2)

table, label = make_data(hard_Z)
compare_imputations(table, label)

Now model-based imputation helps TabPFN and LightGBM as well, not just the linear model. And notice that `native`
TabPFN sits clearly below the `oracle`: even with every source column present, it cannot fully
rebuild a hard column in one forward pass. **TabPFN's internal reconstruction has a complexity
ceiling.** For a genuinely complex recoverable column, explicit model-based imputation still earns
its keep. That is the one real survivor of the classical toolkit.

## Where it actually bites: the failure modes

Native NaN handling has a catch that has nothing to do with TabPFN. A booster learns *where to
route a missing value* from the missingness it saw during training. Two ways that backfires, and
they land very differently for TabPFN.

### The vendor trap: a column missing only at inference

The classic schema or vendor-feed change: a feature was never missing in training, then a feed
drops in production. The booster's routing for "missing" was never learned. Below, the test set
always has 30% of `x0` (a strong predictor) missing, and we sweep how much missingness the model
saw during training.

In [ ]:
# A strong predictor x0; the label depends on it.
rng = np.random.RandomState(0)
X = rng.randn(8000, 10)
label = (rng.rand(8000) < 1 / (1 + np.exp(-(2.5 * X[:, 0] + X[:, 1])))).astype(int)
data = pd.DataFrame(X, columns=[f"x{i}" for i in range(10)])

half = 4000
train_full, test = data.iloc[:half], data.iloc[half:].copy()
y_train, y_test = label[:half], label[half:]

test.loc[rng.rand(half) < 0.30, "x0"] = np.nan      # TEST always has 30% of x0 missing

def train_with_missing(rate):
    """Copy the training set and make x0 missing at the given rate."""
    train = train_full.copy()
    holes = np.random.RandomState(1).rand(half) < rate
    train.loc[holes, "x0"] = np.nan
    return train

models = {
    "TabPFN":   lambda: TabPFNClassifier(random_state=0),
    "LightGBM": lambda: LGBMClassifier(verbose=-1),
    "XGBoost":  lambda: XGBClassifier(tree_method="hist", eval_metric="logloss"),
    "CatBoost": lambda: CatBoostClassifier(verbose=0),
}
train_missing_rates = [0.0, 0.01, 0.05, 0.30]

scores = {}
for rate in train_missing_rates:
    column = f"{int(rate * 100)}% train-missing"
    train = train_with_missing(rate)
    scores[column] = {}
    for model_name, make_model in models.items():
        scores[column][model_name] = auc(make_model(), train, y_train, test, y_test)

pd.DataFrame(scores)

TabPFN is **flat**: its pretrained prior handles a hole whether or not it ever saw one in this
training set. XGBoost and CatBoost **fall off a cliff at 0% train-missing** (roughly minus 0.1
AUC), and even 1% training missingness rescues them. That is exactly the production trap that
passes cross-validation and then quietly tanks when a feed changes.

On real data the same cliff appears (adult, bank-marketing: up to minus 0.12 for XGBoost), and it
bites hardest when the missing column is both important and not reconstructable from the others.
One honest caveat: which booster breaks is unpredictable. LightGBM is robust here, but on other
columns it cliffs the hardest. The only model that never dips is TabPFN.

### Mechanism shift: when the reason changes

The harder failure. A column's missingness is informative, but the rule changes between training
and production. We make a noise column `W` whose only signal is *whether* it is missing: in
training "missing" means label 1, and at test the rule flips so "missing" means label 0.

In [ ]:
# W is pure noise. Only WHETHER it is missing carries signal, and we flip that rule at test.
rng = np.random.RandomState(0)
X = rng.randn(8000, 8)
W = rng.randn(8000)
label = (rng.rand(8000) < 1 / (1 + np.exp(-1.5 * X[:, 0]))).astype(int)

def build(rows, p_missing_if_1, p_missing_if_0, seed):
    """Make W missing with one probability when label==1 and another when label==0."""
    w = W[rows].copy()
    p = np.where(label[rows] == 1, p_missing_if_1, p_missing_if_0)
    holes = np.random.RandomState(seed).rand(len(w)) < p
    w[holes] = np.nan
    columns = [f"x{i}" for i in range(8)] + ["W"]
    return pd.DataFrame(np.column_stack([X[rows], w]), columns=columns)

train_rows = slice(0, 4000)
test_rows = slice(4000, 8000)
y_train, y_test = label[:4000], label[4000:]

train = build(train_rows, p_missing_if_1=0.75, p_missing_if_0=0.25, seed=1)   # missing -> label 1
test_stable = build(test_rows, p_missing_if_1=0.75, p_missing_if_0=0.25, seed=2)   # same rule
test_shifted = build(test_rows, p_missing_if_1=0.25, p_missing_if_0=0.75, seed=3)  # rule flipped

# the same four native-NaN models as the vendor-trap cell
scores = {}
for model_name, make_model in models.items():
    scores[model_name] = {
        "stable": auc(make_model(), train, y_train, test_stable, y_test),
        "shift (rule flips)": auc(make_model(), train, y_train, test_shifted, y_test),
    }

pd.DataFrame(scores).T

When the rule is stable, every model exploits the informative missingness and gains. When it flips,
they all collapse, well below the score they would get by ignoring `W` entirely, and **TabPFN is
right there with the boosters** (having leaned on the signal hardest, it has the most to lose).

This is the trap TabPFN does not escape. Its prior protects you from the *absence* of training
missingness (the vendor trap), not from a *changing* missingness-to-label relationship. That one
you monitor (PSI or adversarial validation on the missing-indicators), you do not model your way
out of it.

## Takeaways

- TabPFN does mean-impute-plus-indicator by default, so most of the classical missing-value toolkit
  is redundant for it: native, mean-fill, sentinel, and hand-built indicators all land within noise.
- The one survivor is **model-based imputation of a genuinely complex recoverable column**, where
  TabPFN's internal reconstruction hits a complexity ceiling.
- The **vendor trap** (a column missing only at inference) is where TabPFN clearly wins: flat, while
  boosters cliff.
- **Mechanism shift** is the honest limit: a changing missingness rule fools TabPFN as much as
  anyone, so you monitor it.

The throughline: native handling models the *routing* of missingness learned from training. TabPFN's
prior covers the routing you never saw, but neither covers a routing that changes underneath you.